# Carregamento da Gold no PostgreSQL

Execute em ordem, com um kernel novo do ambiente virtual do projeto e a Gold já gerada. O PostgreSQL deve estar em execução e a célula de preparação cria o banco se ele ainda não existir. O usuário precisa ter permissão CREATEDB para essa primeira criação. O carregador cria o schema e as tabelas usando as permissões do usuário conectado.

As dependências do projeto estão em `requirements.txt` (`python -m pip install -r requirements.txt`, a partir da raiz do projeto). O carregamento usa Psycopg e COPY; não precisa do driver JDBC.

As cinco tabelas são publicadas em uma única transação. Uma falha desfaz as alterações dessa execução. Reexecuções atualizam as chaves existentes e inserem as novas, preservando outros anos e datas de referência; não removem registros que deixaram de aparecer na entrada. Mudanças no schema exigem uma migração explícita, sem apagar tabelas automaticamente.

O acompanhamento usa somente o estado atual. Os valores permanecem nos arquivos das camadas; não são criados snapshots. Para atualizar os vínculos no PostgreSQL, execute a Gold e depois o Load.


## 1. Imports e localização do projeto e configurar a carga

In [ ]:
import os
import sys
from getpass import getpass
from pathlib import Path

from pyspark.sql import SparkSession
from psycopg import sql


project_directory = Path.cwd().resolve()

while (
    not (project_directory / "pipeline" / "load").is_dir()
    or not (project_directory / "data").is_dir()
):

    if project_directory == project_directory.parent:
        raise FileNotFoundError(
            "Abra o notebook com o diretório de trabalho dentro do projeto Viajando_com_o_dado."
        )

    project_directory = project_directory.parent

sys.path.insert(0, str(project_directory))

from pipeline.shared import execution
from pipeline.load import postgres

GOLD_DIRECTORY = project_directory / "data" / "gold"
YEAR = "2025"
SCHEMA_NAME = "gold"
local_filesystem_jar = (
    project_directory / "pipeline" / "bronze" / "lib" / "hadoop-bare-naked-local-fs-0.1.0.jar"
)

if not GOLD_DIRECTORY.is_dir():
    raise FileNotFoundError(f"Gold não encontrada: {GOLD_DIRECTORY}")

if os.name == "nt" and not local_filesystem_jar.is_file():
    raise FileNotFoundError(f"JAR necessário não encontrado: {local_filesystem_jar}")

print(f"Entrada: {GOLD_DIRECTORY}")
print(f"Ano meteorológico: {YEAR}")
print(f"Schema PostgreSQL: {SCHEMA_NAME}")


In [ ]:
session = execution.run_context(
    project_directory, parameters={"year": str(globals().get("YEAR", "2025"))}
)
state = session.__enter__()
print("Acompanhando a execução atual.")


## 2. Ler o .env e preparar o banco

O arquivo `.env` da raiz é lido automaticamente. Preencha `PGPASSWORD` nesse arquivo e mantenha-o fora do versionamento; `.env.example` mostra a estrutura sem credenciais. Variáveis já definidas no ambiente têm prioridade.

São usadas as variáveis `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, `PGPASSWORD` e `PGSSLMODE`. Os padrões abaixo apontam para uma instalação local; ajuste o banco e o usuário conforme sua instalação. Para um servidor remoto, configure o modo SSL exigido por ele.

Se a senha não estiver no ambiente, a célula solicita sua digitação oculta. Ela permanece somente na memória da sessão, sem ser salva no notebook. As configurações também poderão ser fornecidas por variáveis de ambiente no futuro executor da pipeline.

In [ ]:
postgres.load_environment()

os.environ.setdefault("PGHOST", "localhost")
os.environ.setdefault("PGPORT", "5432")
os.environ.setdefault("PGDATABASE", "viajando_com_o_dado")
os.environ.setdefault("PGUSER", "postgres")
os.environ.setdefault("PGSSLMODE", "prefer")

if not os.environ.get("PGPASSWORD"):
    os.environ["PGPASSWORD"] = getpass("Senha do PostgreSQL: ")

# Preserva o banco se ele já existir; a primeira criação exige permissão CREATEDB.
database_created = postgres.create_database()
print("Banco criado." if database_created else "Banco já existente; conteúdo preservado.")

# Verifica o acesso sem criar ou modificar tabelas.
with postgres.get_connection() as connection:

    with connection.cursor() as cursor:

        cursor.execute("SELECT current_database(), current_user")
        database_name, database_user = cursor.fetchone()

print(f"Conexão disponível: banco {database_name}; usuário {database_user}.")


## 3. Iniciar o Spark para ler os Parquets da Gold

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable

spark_builder = (
    SparkSession.builder.appName("Viajando com o dado - Load PostgreSQL")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.pyspark.python", sys.executable)
)

if os.name == "nt":
    spark_builder = (
        spark_builder.config("spark.driver.extraClassPath", str(local_filesystem_jar))
        .config("spark.executor.extraClassPath", str(local_filesystem_jar))
        .config(
            "spark.hadoop.fs.file.impl", "com.globalmentor.apache.hadoop.fs.BareLocalFileSystem"
        )
    )

spark = spark_builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Versão do Spark: {spark.version}")


## 4. Carregar as cinco tabelas

| Saída da Gold | Tabela no PostgreSQL | Chave primária |
|---|---|---|
| Municípios IBGE | `gold.municipalities` | `municipality_id` |
| População IBGE | `gold.population` | `municipality_id`, `reference_date` |
| Geocoding | `gold.geocoding` | `municipality_id` |
| Open-Meteo diário | `gold.open_meteo_daily` | `municipality_id`, `observation_date` |
| INMET diário | `gold.inmet_daily` | `municipality_id`, `observation_date` |

O prefixo `gold` acompanha `SCHEMA_NAME`. Datas, números, booleanos e ausências mantêm seus tipos. Listas, origens e estruturas são armazenadas em JSONB. O cadastro é carregado primeiro, para atender às chaves estrangeiras das outras quatro tabelas.

Esta célula realiza a gravação no banco configurado. Ela só retorna o resumo após a confirmação da transação completa. As contagens representam registros processados nesta execução, incluindo registros atualizados.

In [ ]:
load_summary = postgres.load_gold(
    spark,
    GOLD_DIRECTORY,
    year=YEAR,
    schema_name=SCHEMA_NAME,
)

for table_name, record_count in load_summary.items():

    print(f"{SCHEMA_NAME}.{table_name}: {record_count} registros processados.")


## 5. Conferir as tabelas no PostgreSQL

O total no banco pode ser maior que o processado nesta execução, porque anos e datas anteriores são preservados. A chave primária impede duplicar a mesma capital na mesma data. A carga não faz preenchimento de medições ausentes.

In [ ]:
with postgres.get_connection() as connection:

    with connection.cursor() as cursor:

        for table_name in load_summary:

            cursor.execute(
                sql.SQL("SELECT count(*) FROM {}").format(sql.Identifier(SCHEMA_NAME, table_name))
            )
            total_rows = cursor.fetchone()[0]

            print(f"{SCHEMA_NAME}.{table_name}: {total_rows} registros no banco.")


## 6. Encerrar o Spark

As conexões PostgreSQL das células anteriores são encerradas pelos próprios blocos `with`.

In [ ]:
execution.finish_notebook(session, spark)
